## 05 – Hyper‑parameter Tuning (Random Forest)

Ce notebook réalise une recherche d’hyper‑paramètres via `GridSearchCV` (entraînement **uniquement** sur le jeu d’entraînement) et sauvegarde le meilleur modèle.

In [1]:
# Imports standard
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

# Scikit‑learn & imbalanced‑learn
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.metrics import f1_score
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# -------------------------------------------------------------------
# 1️⃣ Charger les données engineered (Parquet)
# -------------------------------------------------------------------
PROJECT_ROOT = Path('..')
DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'

df = pd.read_parquet(DATA_DIR / 'dataset_engineered.parquet')
# Suppression éventuelle de colonnes non‑features (ex. l'URL)
if 'url' in df.columns:
    df = df.drop(columns=['url'])

X = df.drop(columns=['is_phishing'])
y = df['is_phishing']
print('Shape X:', X.shape)
print('Classes distribution:', np.bincount(y))

Shape X: (11000, 22)
Classes distribution: [9000 2000]


In [3]:
# -------------------------------------------------------------------
# 2️⃣ Création des splits Train / Validation / Test (70/15/15)
# -------------------------------------------------------------------
# Première séparation : Train (70 %) vs Temp (30 %)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
# Deuxième séparation du Temp en Validation et Test (chacun ≈15 %)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)
print('Train size :', X_train.shape[0])
print('Validation size :', X_valid.shape[0])
print('Test size :', X_test.shape[0])

Train size : 7700
Validation size : 1650
Test size : 1650


In [ ]:
# -------------------------------------------------------------------
# 3️⃣ Pipeline de base Random Forest (sans ré‑équilibrage)
# -------------------------------------------------------------------
preprocessor_path = MODELS_DIR / 'preprocessor.joblib'
preprocessor = joblib.load(preprocessor_path)

rf_best_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    ))
] )

In [5]:
# -------------------------------------------------------------------
# 4️⃣ Définition de la grille d’hyper‑paramètres
# -------------------------------------------------------------------
param_grid = {
    'classifier__n_estimators': [200, 500],
    'classifier__max_depth': [10, 20, None],
    'classifier__min_samples_leaf': [1, 2, 5],
    'classifier__max_features': ['sqrt']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=rf_best_pipeline,
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

In [ ]:
# -------------------------------------------------------------------
# 5️⃣ Entraînement du GridSearch **sur le jeu d'entraînement uniquement**
# -------------------------------------------------------------------
grid_search.fit(X_train, y_train)
print(' Meilleur F1 (CV) :', grid_search.best_score_)
print(' Meilleurs hyper‑paramètres :', grid_search.best_params_)

Fitting 5 folds for each of 18 candidates, totalling 90 fits
🔎 Meilleur F1 (CV) : 0.9960796638531827
🔧 Meilleurs hyper‑paramètres : {'classifier__max_depth': 20, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 500}


In [7]:
# -------------------------------------------------------------------
# 6️⃣ Tableau des 10 meilleures combinaisons
# -------------------------------------------------------------------
results_df = pd.DataFrame(grid_search.cv_results_)
top10 = results_df.sort_values('mean_test_score', ascending=False).head(10)
display(top10[['params', 'mean_test_score', 'std_test_score']])

,params,mean_test_score,std_test_score
13,"{'classifier__max_depth': None, 'classifier__m...",0.996080,0.003058
7,"{'classifier__max_depth': 20, 'classifier__max...",0.996080,0.003058
1,"{'classifier__max_depth': 10, 'classifier__max...",0.995719,0.002136
0,"{'classifier__max_depth': 10, 'classifier__max...",0.995719,0.002136
12,"{'classifier__max_depth': None, 'classifier__m...",0.995365,0.003301
6,"{'classifier__max_depth': 20, 'classifier__max...",0.995365,0.003301
2,"{'classifier__max_depth': 10, 'classifier__max...",0.994662,0.003363
14,"{'classifier__max_depth': None, 'classifier__m...",0.994662,0.003363
8,"{'classifier__max_depth': 20, 'classifier__max...",0.994662,0.003363
3,"{'classifier__max_depth': 10, 'classifier__max...",0.994659,0.002511


In [ ]:
# -------------------------------------------------------------------
# 7️⃣ Sauvegarde du meilleur pipeline (nom attendu par le projet)
# -------------------------------------------------------------------
final_path = MODELS_DIR / 'final_model.joblib'
final_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(grid_search.best_estimator_, final_path)
print(' Modèle final sauvegardé à', final_path)

✅ Modèle final sauvegardé à ..\models\final_model.joblib
